# Acceso a Datos — Manejo de Ficheros en Python

**Módulo:** Acceso a Datos — 2º DAM (Desarrollo de Aplicaciones Multiplataforma)

Este cuaderno recoge, con ejemplos prácticos y **100% ejecutables en Python**, los contenidos del
bloque de *Manejo de Ficheros*:

1. Clases/módulos para la gestión de ficheros y directorios (creación, borrado, copia, movimiento...)
2. Formas de acceso a un fichero: secuencial vs. aleatorio
3. Flujos (streams): basados en bytes y basados en caracteres
4. Clases para gestionar flujos de datos desde/hacia ficheros
5. Operaciones básicas sobre ficheros
6. Trabajo con ficheros XML: parsers (DOM, SAX, ElementTree) y *binding*
7. Librerías para convertir XML a otros formatos
8. Excepciones: detección y tratamiento

---

> 💡 Aunque el módulo de Acceso a Datos suele impartirse con Java, **todos los conceptos son
> transferibles a cualquier lenguaje**. En este cuaderno se explican exactamente los mismos
> contenidos, pero usando la librería estándar de **Python** (`os`, `pathlib`, `io`, `struct`,
> `pickle`, `xml.dom.minidom`, `xml.sax`, `xml.etree.ElementTree`...) y alguna librería adicional muy
> extendida (`lxml`, `xmltodict`). Se indican en todo momento los nombres equivalentes en Java para
> quien quiera relacionar ambos mundos.


## 1. Módulos para la gestión de ficheros y directorios

Python ofrece varias formas de trabajar con ficheros y directorios como *entidades del sistema de
archivos* (su existencia, ubicación, metadatos...), sin entrar todavía en su contenido:

| Módulo | Estilo | Equivalente conceptual en Java |
|---|---|---|
| `os` / `os.path` | Funciones sueltas, con rutas como cadenas de texto | `java.io.File` (API clásica) |
| `pathlib.Path` | **Orientado a objetos**, la ruta es un objeto con métodos | `java.nio.file.Path` / `Files` (NIO.2) |
| `shutil` | Operaciones de "alto nivel": copiar, mover, borrar árboles completos | Utilidades añadidas sobre `Files` (`Files.copy`, `Files.move`...) |

En proyectos modernos se recomienda **`pathlib`**, igual que en Java se recomienda NIO.2 frente a la
API clásica de `File`. Aun así, es habitual encontrar `os`/`os.path` en código existente.

### 1.1. `pathlib.Path`: crear, consultar, renombrar y borrar


In [2]:
from pathlib import Path

# Crear un directorio
carpeta = Path("datos_demo")
carpeta.mkdir(exist_ok=True)
print("Directorio creado:", carpeta.resolve())

# Crear un fichero (vacío) dentro del directorio
fichero = carpeta / "notas.txt"
fichero.touch()
print("Fichero creado:", fichero.name)

# Consultar metadatos
print("¿Existe?", fichero.exists())
print("¿Es directorio?", carpeta.is_dir())
print("¿Es fichero?", fichero.is_file())
print("Tamaño (bytes):", fichero.stat().st_size)

# Renombrar / mover (rename también sirve para mover si cambia la ruta)
copia = carpeta / "notas_copia.txt"
fichero.rename(copia)
print("Renombrado a:", copia.name)

# Listar el contenido de un directorio
print("Contenido del directorio:")
for f in carpeta.iterdir():
    print(" -", f.name)

# Borrado (unlink solo borra FICHEROS; rmdir solo borra directorios VACÍOS)
#copia.unlink()
#carpeta.rmdir()
#print("¿Sigue existiendo el directorio?", carpeta.exists())


Directorio creado: C:\Users\agmze\OneDrive - Educacyl\2026-2027 IES Julián Marías\Modulos\AAD\Materiales\Tema 1\datos_demo
Fichero creado: notas.txt
¿Existe? True
¿Es directorio? True
¿Es fichero? True
Tamaño (bytes): 0
Renombrado a: notas_copia.txt
Contenido del directorio:
 - notas_copia.txt


> **Importante:** al igual que en Java `File.delete()` solo borra directorios **vacíos**,
> `Path.rmdir()` exige que el directorio esté vacío y `Path.unlink()` solo borra ficheros. Para borrar
> un árbol completo de directorios (con contenido) se usa `shutil.rmtree()`.

### 1.2. `shutil`: copiar, mover y borrar árboles completos


In [2]:
import shutil
from pathlib import Path

base = Path("shutil_area")
base.mkdir(exist_ok=True)

origen = base / "original.txt"
origen.write_text("Contenido de ejemplo con pathlib/shutil\nSegunda línea\n", encoding="utf-8")

# Copiar (shutil.copy conserva el contenido; copy2 conserva también metadatos como la fecha)
copia = base / "copia.txt"
shutil.copy2(origen, copia)
print("Copiado ->", copia.name)

# Mover (shutil.move funciona incluso entre discos distintos, a diferencia de Path.rename)
destino = base / "subcarpeta"
destino.mkdir(exist_ok=True)
movido = shutil.move(str(copia), str(destino / "copia_movida.txt"))
print("Movido ->", movido)

print("¿Es legible?", origen.is_file() and origen.stat().st_size > 0)
print("Tamaño:", origen.stat().st_size, "bytes")

print(f"Árbol de {base}:")
for p in sorted(base.rglob("*")):
    print("  ", p.relative_to(base))

# Borrado de un árbol completo de directorios (aunque tenga contenido)
shutil.rmtree(base)
print("¿Sigue existiendo?", base.exists())


Copiado -> copia.txt
Movido -> shutil_area/subcarpeta/copia_movida.txt
¿Es legible? True
Tamaño: 55 bytes
Árbol de shutil_area:
   original.txt
   subcarpeta
   subcarpeta/copia_movida.txt
¿Sigue existiendo? False


**Otras operaciones frecuentes:** `Path.exists()`, `os.path.getsize()`, `tempfile.NamedTemporaryFile()`
(equivalente a `Files.createTempFile()`), `Path.read_text()` / `Path.write_text()` (lectura/escritura
directa sin abrir manualmente el fichero) y `os.walk()` / `Path.rglob()` (recorrido recursivo de un
árbol de directorios, equivalente a `Files.walk()` / `Files.walkFileTree()`).


## 2. Formas de acceso a un fichero

A la hora de leer o escribir el **contenido** de un fichero, existen dos modelos:

| Forma de acceso | Cómo funciona | Ventajas | Inconvenientes | Campos de aplicación |
|---|---|---|---|---|
| **Secuencial** | Se lee/escribe de principio a fin, sin poder "saltar" | Simple, eficiente para lectura completa, bajo consumo de memoria | Para llegar al final hay que pasar por todo lo anterior; lento si solo necesitamos un dato concreto | Ficheros de texto, logs, ficheros de configuración, *streaming* de datos |
| **Aleatorio (directo)** | Se puede posicionar el "cursor" en cualquier byte del fichero y leer/escribir desde ahí | Acceso muy rápido a un registro concreto sin recorrer el resto | Requiere que los registros tengan **tamaño fijo** (o un índice aparte); código más complejo | Bases de datos ligeras, ficheros de registros de tamaño fijo (ej. fichas de alumnos), estructuras tipo tabla |

En Python, **todo objeto fichero abierto en modo binario o texto admite acceso aleatorio** mediante
los métodos `seek(posicion)` (mover el cursor) y `tell()` (consultar la posición actual). Esto
equivale a la clase `java.io.RandomAccessFile`, salvo que en Python no existe una clase distinta: es
el mismo objeto devuelto por `open()`, simplemente usado de otra manera.

Para simular registros de tamaño fijo (como en Java) usamos el módulo `struct`, que empaqueta valores
Python en secuencias de bytes con un formato binario preciso.


In [3]:
import struct
from pathlib import Path

# Diseñamos un registro de tamaño FIJO:
#   nombre: 20 bytes (cadena rellenada con espacios)
#   nota:   8 bytes (double / float de precisión doble)
FORMATO_REGISTRO = "20sd"                     # "20s" = cadena de 20 bytes, "d" = double
LONG_REGISTRO = struct.calcsize(FORMATO_REGISTRO)

def escribir_registro(f, indice, nombre, nota):
    f.seek(indice * LONG_REGISTRO)             # saltamos directamente a la posición del registro
    datos = struct.pack(FORMATO_REGISTRO, nombre.encode("utf-8").ljust(20), nota)
    f.write(datos)

def leer_registro(f, indice):
    f.seek(indice * LONG_REGISTRO)
    datos = f.read(LONG_REGISTRO)
    nombre_bytes, nota = struct.unpack(FORMATO_REGISTRO, datos)
    nombre = nombre_bytes.decode("utf-8").strip()
    print(f"Registro {indice} -> {nombre} | nota: {nota:.2f}")

ruta = Path("alumnos.dat")
with open(ruta, "wb+") as f:                    # "wb+" = binario, lectura Y escritura
    escribir_registro(f, 0, "Ana", 8.5)
    escribir_registro(f, 1, "Luis", 6.0)
    escribir_registro(f, 2, "Marta", 9.25)

    # Accedemos DIRECTAMENTE al registro 2 sin pasar por el 0 y el 1
    leer_registro(f, 2)
    leer_registro(f, 0)

    # Modificamos solo la nota de Luis (registro 1) sin reescribir el fichero completo
    f.seek(1 * LONG_REGISTRO + 20)              # 20 bytes del nombre, luego viene la nota
    f.write(struct.pack("d", 7.5))
    leer_registro(f, 1)

    f.seek(0, 2)                                # seek(0, 2) = mover el cursor al final del fichero
    print("Tamaño total del fichero:", f.tell(), f"bytes (3 x {LONG_REGISTRO})")

ruta.unlink()


Registro 2 -> Marta | nota: 9.25
Registro 0 -> Ana | nota: 8.50
Registro 1 -> Luis | nota: 6.00
Tamaño total del fichero: 96 bytes (3 x 32)


Observa que hemos podido **leer el registro 2 sin leer antes el 0 y el 1**, y **modificar solo la
nota de un registro** sin reescribir el fichero completo: esa es la ventaja fundamental del acceso
aleatorio frente al secuencial. El segundo argumento de `seek(desplazamiento, origen)` indica desde
dónde se cuenta: `0` = inicio del fichero (por defecto), `1` = posición actual, `2` = final del
fichero.


## 3. Flujos (*Streams*)

Un **flujo** es un canal de comunicación para transmitir datos entre el programa y un origen/destino
(fichero, red, memoria...). Python distingue, igual que Java, dos grandes familias, seleccionadas con
el **modo de apertura** que se pasa a `open()`:

| | Basados en **bytes** (modo `"b"`) | Basados en **caracteres** (modo por defecto, texto) |
|---|---|---|
| Ejemplo de apertura | `open(ruta, "rb")` / `open(ruta, "wb")` | `open(ruta, "r")` / `open(ruta, "w")` |
| Tipo Python devuelto | `bytes` en cada lectura | `str` en cada lectura |
| Uso típico | Datos **binarios**: imágenes, audio, ejecutables, objetos serializados | **Texto**: ficheros `.txt`, `.csv`, `.xml`, `.json`... |
| Ventaja | Control total byte a byte, válido para cualquier tipo de dato | Gestiona automáticamente la codificación de caracteres (parámetro `encoding`) |
| Inconveniente | Si el contenido es texto, hay que tratar la codificación "a mano" | No es apto para datos binarios (puede corromper bytes) |
| Equivalente Java | `InputStream` / `OutputStream` | `Reader` / `Writer` |

En Python **no hay clases separadas** como en Java (`FileInputStream` vs. `FileReader`): la misma
función `open()` devuelve un tipo u otro de objeto según el modo (`"b"` presente o no).


In [4]:
from pathlib import Path

texto = "Café con leche y ñoquis"
ruta = Path("bytes.dat")

# Escritura en modo BINARIO: convertimos el str a bytes explícitamente con UTF-8
with open(ruta, "wb") as f:
    f.write(texto.encode("utf-8"))

print("Bytes escritos en el fichero:")
with open(ruta, "rb") as f:
    datos = f.read()
    for b in datos:
        print(b, end=" ")
    print(f"\nTotal de bytes leídos: {len(datos)}")

# Reconstrucción a partir de los bytes leídos
with open(ruta, "rb") as f:
    contenido = f.read()
print("Reconstruido:", contenido.decode("utf-8"))

ruta.unlink()


Bytes escritos en el fichero:
67 97 102 195 169 32 99 111 110 32 108 101 99 104 101 32 121 32 195 177 111 113 117 105 115 
Total de bytes leídos: 25
Reconstruido: Café con leche y ñoquis


Fíjate en que **"Café con leche y ñoquis" ocupa 25 bytes**, no 23 caracteres: las letras
acentuadas (`é`, `ñ`) ocupan **2 bytes** en UTF-8. Este es precisamente el problema que resuelven los
flujos de caracteres: trabajar en la unidad "carácter" en lugar de en "byte".


In [5]:
from pathlib import Path

texto = "Café con leche y ñoquis"
ruta = Path("chars.txt")

# Apertura en modo TEXTO: se indica explícitamente el encoding a usar (buena práctica)
with open(ruta, "w", encoding="utf-8") as f:
    f.write(texto)

print("Lectura carácter a carácter:")
with open(ruta, "r", encoding="utf-8") as f:
    contenido = f.read()
    print(f"Total de caracteres leídos: {len(contenido)} (frente a 25 bytes en UTF-8)")

# readline(): lee una única línea (equivalente a BufferedReader.readLine() en Java)
with open(ruta, "r", encoding="utf-8") as f:
    linea = f.readline()
    print("Línea leída con readline():", linea)

# Modo "a" (append): añade contenido al final sin borrar lo anterior
with open(ruta, "a", encoding="utf-8") as f:
    f.write("\n")
    f.write(f"Nota media: {7.8:.2f}\n")

with open(ruta, "r", encoding="utf-8") as f:
    print("Contenido final del fichero:")
    for linea in f:                     # iterar directamente sobre el fichero = leer línea a línea
        print(" ", linea.rstrip("\n"))

ruta.unlink()


Lectura carácter a carácter:
Total de caracteres leídos: 23 (frente a 25 bytes en UTF-8)
Línea leída con readline(): Café con leche y ñoquis
Contenido final del fichero:
  Café con leche y ñoquis
  Nota media: 7.80


## 4. El módulo `io`: jerarquía de flujos

Python organiza los flujos en el módulo `io` con una jerarquía muy similar, conceptualmente, a la de
Java. `open()` es en realidad una función "fábrica" que, según el modo, devuelve un objeto u otro:

```
io.IOBase (base abstracta)
 ├─ io.RawIOBase            -> acceso "en crudo" al sistema operativo (poco usado directamente)
 ├─ io.BufferedIOBase        equivalente a BufferedInputStream/OutputStream de Java
 │   ├─ BufferedReader / BufferedWriter   (usados internamente por open(..., "rb"/"wb"))
 │   └─ BufferedRandom                    (lectura Y escritura en binario, con buffer)
 └─ io.TextIOBase             equivalente a Reader/Writer de Java
     └─ TextIOWrapper                     (lo que devuelve open() en modo texto, "r"/"w")
```

Para tipos de datos más allá de texto/bytes crudos, Python usa **módulos** en lugar de clases
decoradoras adicionales:

| Necesidad | Módulo/función Python | Equivalente en Java |
|---|---|---|
| Leer/escribir tipos primitivos en binario con formato preciso | `struct.pack()` / `struct.unpack()` | `DataOutputStream` / `DataInputStream` |
| Serializar objetos Python completos | `pickle.dump()` / `pickle.load()` | `ObjectOutputStream` / `ObjectInputStream` |
| Flujo en memoria (sin fichero real) | `io.BytesIO()` / `io.StringIO()` | `ByteArrayOutputStream` / `StringWriter` |


In [6]:
import pickle
import struct
from dataclasses import dataclass, field
from pathlib import Path

@dataclass
class Alumno:
    nombre: str
    nota: float
    password_temporal: str = field(repr=False, compare=False)

    def __getstate__(self):
        # Igual que el "transient" de Java: excluimos password_temporal de la serialización
        estado = self.__dict__.copy()
        estado["password_temporal"] = None
        return estado

lista = [Alumno("Ana", 8.5, "1234"), Alumno("Luis", 6.0, "abcd")]

# --- pickle: serialización de objetos completos ---
ruta_pickle = Path("alumnos.pkl")
with open(ruta_pickle, "wb") as f:
    pickle.dump(lista, f)
print("Objeto serializado en alumnos.pkl")

with open(ruta_pickle, "rb") as f:
    recuperada = pickle.load(f)
print("Objetos recuperados:")
for a in recuperada:
    print(f"  {a.nombre} ({a.nota}) password_temporal={a.password_temporal}")

# --- struct: escritura/lectura de tipos primitivos en binario, con formato exacto ---
ruta_bin = Path("datos.bin")
with open(ruta_bin, "wb") as f:
    nombre_bytes = "Ana".encode("utf-8")
    f.write(struct.pack("B", len(nombre_bytes)))   # 1 byte con la longitud del nombre
    f.write(nombre_bytes)                            # el propio nombre
    f.write(struct.pack("d", 8.5))                    # nota (double)
    f.write(struct.pack("i", 2026))                   # año (int)

with open(ruta_bin, "rb") as f:
    (longitud,) = struct.unpack("B", f.read(1))
    nombre = f.read(longitud).decode("utf-8")
    (nota,) = struct.unpack("d", f.read(8))
    (anio,) = struct.unpack("i", f.read(4))
    print(f"struct -> {nombre}, {nota}, {anio}")

ruta_pickle.unlink()
ruta_bin.unlink()


Objeto serializado en alumnos.pkl
Objetos recuperados:
  Ana (8.5) password_temporal=None
  Luis (6.0) password_temporal=None
struct -> Ana, 8.5, 2026


Observa dos detalles importantes:

- El campo `password_temporal` se excluye de la serialización mediante `__getstate__` (equivale al
  modificador `transient` de Java): al recuperar el objeto aparece como `None`.
- Con `struct`, el orden y el formato de lectura deben coincidir **exactamente** con los usados al
  escribir (igual que con `DataInputStream`/`DataOutputStream` en Java): aquí primero escribimos la
  *longitud* del nombre en 1 byte, precisamente porque, a diferencia de Java, `struct` no reserva un
  tamaño fijo para cadenas salvo que se indique explícitamente (`"20s"` como en el apartado 2).


## 5. Operaciones básicas sobre ficheros

Independientemente del modo de apertura, cualquier operación sobre el **contenido** de un fichero
sigue siempre el mismo patrón:

1. **Abrir** el fichero (`open()`), asociándolo a una ruta concreta.
2. **Leer y/o escribir** datos.
3. **Cerrar** el fichero (`close()`), liberando el recurso del sistema operativo.

La sentencia **`with`** (*context manager*) garantiza el cierre automático del fichero incluso si se
produce una excepción dentro del bloque — es el equivalente exacto del `try-with-resources` de Java.
**Siempre** se debe usar `with` en lugar de llamar a `close()` manualmente.


In [7]:
from pathlib import Path

origen = Path("original.txt")
destino = Path("copia_manual.txt")
origen.write_text("Línea 1\nLínea 2\nLínea 3\n", encoding="utf-8")

# Copia manual línea a línea usando abrir -> leer/escribir -> cerrar (automático con "with")
with open(origen, "r", encoding="utf-8") as lector, \
     open(destino, "w", encoding="utf-8") as escritor:

    for num_linea, linea in enumerate(lector, start=1):
        escritor.write(f"{num_linea}: {linea}")
# <-- aquí Python cierra AUTOMÁTICAMENTE lector y escritor, aunque hubiera habido una excepción

print("Copia con numeración de líneas:")
print(destino.read_text(encoding="utf-8"))

origen.unlink()
destino.unlink()


Copia con numeración de líneas:
1: Línea 1
2: Línea 2
3: Línea 3



### Antes de `with` (equivalente al `try/finally` manual de Java)

Por completitud, así se garantizaba el cierre de un fichero **sin** `with` — código que todavía es
importante saber leer:

```python
lector = None
try:
    lector = open("fichero.txt", "r", encoding="utf-8")
    # ... trabajar con el fichero ...
except IOError as e:
    print("Error de E/S:", e)
finally:
    if lector is not None:
        lector.close()
```

El `with` moderno evita este anidamiento de `try`/`finally` y es mucho menos propenso a errores
(además de funcionar con **cualquier** objeto que implemente el protocolo `__enter__`/`__exit__`, no
solo con ficheros).

### Principales métodos de lectura/escritura

| Método | Efecto |
|---|---|
| `f.read()` | Lee **todo** el contenido restante de una vez |
| `f.read(n)` | Lee como máximo `n` caracteres/bytes |
| `f.readline()` | Lee **una** línea (incluye el salto de línea final) |
| `f.readlines()` | Lee **todas** las líneas y las devuelve como una lista de `str` |
| `for linea in f:` | Itera línea a línea (la forma más eficiente en memoria para ficheros grandes) |
| `f.write(texto)` | Escribe una cadena (no añade salto de línea automáticamente) |
| `f.writelines(lista)` | Escribe una lista de cadenas seguidas |


## 6. Trabajo con ficheros XML: *parsers* y *binding*

Partimos de este fichero XML de ejemplo (`alumnos.xml`) que usaremos en todos los ejemplos de esta
sección:

```xml
<?xml version="1.0" encoding="UTF-8"?>
<alumnos>
    <alumno id="1">
        <nombre>Ana García</nombre>
        <nota>8.5</nota>
    </alumno>
    <alumno id="2">
        <nombre>Luis Pérez</nombre>
        <nota>6.0</nota>
    </alumno>
    <alumno id="3">
        <nombre>Marta Ruiz</nombre>
        <nota>9.25</nota>
    </alumno>
</alumnos>
```

Existen dos grandes enfoques para trabajar con XML:

- **Analizadores sintácticos (*parsers*)**: leen el XML y lo convierten en una estructura genérica
  (árbol de nodos o secuencia de eventos) que el programador recorre "a mano". Python incluye **DOM**
  (`xml.dom.minidom`), **SAX** (`xml.sax`) y una API intermedia muy popular llamada
  **`ElementTree`** (equivalente conceptual a StAX).
- **Vinculación (*binding*)**: el XML se convierte **directamente en objetos Python** (y viceversa) de
  forma automática. Python no incluye un "JAXB" en su librería estándar, pero el patrón se consigue
  fácilmente combinando `dataclasses` con `ElementTree`, o usando librerías como `xmltodict`.


In [8]:
from pathlib import Path

xml_alumnos = '''<?xml version="1.0" encoding="UTF-8"?>
<alumnos>
    <alumno id="1">
        <nombre>Ana García</nombre>
        <nota>8.5</nota>
    </alumno>
    <alumno id="2">
        <nombre>Luis Pérez</nombre>
        <nota>6.0</nota>
    </alumno>
    <alumno id="3">
        <nombre>Marta Ruiz</nombre>
        <nota>9.25</nota>
    </alumno>
</alumnos>
'''
Path("alumnos.xml").write_text(xml_alumnos, encoding="utf-8")
print("Fichero alumnos.xml preparado para los siguientes ejemplos.")


Fichero alumnos.xml preparado para los siguientes ejemplos.


### 6.1. DOM (*Document Object Model*) — `xml.dom.minidom`

Carga **todo el documento en memoria** como un árbol de objetos navegable en cualquier dirección
(hacia arriba, abajo, adelante o atrás) y **modificable**.


In [9]:
from xml.dom.minidom import parse

doc = parse("alumnos.xml")
alumnos = doc.getElementsByTagName("alumno")
print("Alumnos encontrados:", len(alumnos))

suma_notas = 0
for alumno in alumnos:
    id_alumno = alumno.getAttribute("id")
    nombre = alumno.getElementsByTagName("nombre")[0].firstChild.data
    nota = float(alumno.getElementsByTagName("nota")[0].firstChild.data)
    suma_notas += nota
    print(f"  [{id_alumno}] {nombre} -> {nota:.2f}")

print(f"Media de notas: {suma_notas / len(alumnos):.2f}")

# DOM permite MODIFICAR el árbol en memoria (aquí añadimos un alumno nuevo)
nuevo = doc.createElement("alumno")
nuevo.setAttribute("id", "4")

nombre_el = doc.createElement("nombre")
nombre_el.appendChild(doc.createTextNode("Pedro Soto"))
nuevo.appendChild(nombre_el)

nota_el = doc.createElement("nota")
nota_el.appendChild(doc.createTextNode("7.0"))
nuevo.appendChild(nota_el)

doc.documentElement.appendChild(nuevo)
print("Nodo añadido en memoria. Total ahora:",
      len(doc.getElementsByTagName("alumno")))
# (para volcar el árbol modificado de vuelta a un fichero: doc.writexml(...) o doc.toxml())


Alumnos encontrados: 3
  [1] Ana García -> 8.50
  [2] Luis Pérez -> 6.00
  [3] Marta Ruiz -> 9.25
Media de notas: 7.92
Nodo añadido en memoria. Total ahora: 4


### 6.2. SAX (*Simple API for XML*) — `xml.sax`

Procesa el documento **secuencialmente**, generando **eventos** (inicio de etiqueta, texto, fin de
etiqueta...) que se capturan mediante *callbacks* (una subclase de `ContentHandler`). **No** construye
ningún árbol en memoria: es ideal para ficheros muy grandes, pero no permite retroceder ni modificar
el XML.


In [10]:
import xml.sax

class ManejadorAlumnos(xml.sax.ContentHandler):
    def __init__(self):
        super().__init__()
        self.elemento_actual = ""
        self.nombre = ""
        self.total = 0
        self.suma = 0.0

    def startElement(self, name, attrs):
        self.elemento_actual = name
        if name == "alumno":
            print(f"Alumno id={attrs.get('id')} -> ", end="")
            self.total += 1

    def characters(self, content):
        valor = content.strip()
        if not valor:
            return
        if self.elemento_actual == "nombre":
            self.nombre = valor
        elif self.elemento_actual == "nota":
            nota = float(valor)
            self.suma += nota
            print(f"{self.nombre} ({nota})")

    def endDocument(self):
        print(f"SAX terminado. Media: {self.suma / self.total:.2f}")

# SAX recorre el documento evento a evento, SIN cargarlo entero en memoria
xml.sax.parse("alumnos.xml", ManejadorAlumnos())


Alumno id=1 -> Ana García (8.5)
Alumno id=2 -> Luis Pérez (6.0)
Alumno id=3 -> Marta Ruiz (9.25)
SAX terminado. Media: 7.92


### 6.3. `ElementTree` (equivalente conceptual a StAX)

`xml.etree.ElementTree` es la API "moderna" de Python para XML: por defecto funciona como DOM (carga
el árbol completo con `ET.parse()`), pero también ofrece `ET.iterparse()`, que procesa el documento de
forma **incremental**, evento a evento, sin cargarlo entero en memoria — igual que hace StAX en Java
(modelo ***pull***: el programa pide el siguiente evento).


In [11]:
import xml.etree.ElementTree as ET

print("--- Modo árbol completo (como DOM) ---")
arbol = ET.parse("alumnos.xml")
raiz = arbol.getroot()
for alumno in raiz.findall("alumno"):
    print(" ", alumno.get("id"), "->", alumno.find("nombre").text, alumno.find("nota").text)

print("\n--- Modo streaming con iterparse() (como StAX) ---")
elemento_actual = ""
for evento, elem in ET.iterparse("alumnos.xml", events=("start", "end")):
    if evento == "start" and elem.tag == "alumno":
        print(f"id={elem.get('id')} ", end="")
    elif evento == "end" and elem.tag == "nombre":
        print(f"{elem.text} -> ", end="")
    elif evento == "end" and elem.tag == "nota":
        print(elem.text)
    elif evento == "end":
        elem.clear()   # libera memoria del nodo ya procesado (clave para ficheros grandes)

print("ElementTree.iterparse(): modelo pull, bajo consumo de memoria, similar a StAX.")


--- Modo árbol completo (como DOM) ---
  1 -> Ana García 8.5
  2 -> Luis Pérez 6.0
  3 -> Marta Ruiz 9.25

--- Modo streaming con iterparse() (como StAX) ---
id=1 Ana García -> 8.5
id=2 Luis Pérez -> 6.0
id=3 Marta Ruiz -> 9.25
ElementTree.iterparse(): modelo pull, bajo consumo de memoria, similar a StAX.


### Comparativa de los analizadores XML

| | **DOM** (`minidom`) | **SAX** (`xml.sax`) | **ElementTree** (`iterparse`) |
|---|---|---|---|
| Modelo | Árbol completo en memoria | Eventos (*push*) | Eventos (*pull*) |
| Consumo de memoria | Alto (todo el documento) | Bajo | Bajo (si se usa `elem.clear()`) |
| Navegación | Libre (adelante/atrás) | Solo hacia adelante | Solo hacia adelante, controlado por el programa |
| Modificación del XML | Sí | No | Sí, en modo árbol completo (`ET.parse`) |
| Facilidad de uso | Media | Baja (requiere subclase *handler*) | **Alta** — es la más usada en Python actual |
| Cuándo usarlo | Ficheros pequeños/medianos que hay que consultar o modificar varias veces | Ficheros muy grandes, procesado de un solo paso | Uso general; combina lo mejor de ambos mundos |

### 6.4. *Binding*: mapear XML a objetos Python

Python no incluye en su librería estándar un equivalente directo a JAXB, pero el patrón se reproduce
fácilmente combinando `dataclasses` con `ElementTree` (mapeo manual) o usando `xmltodict`
(conversión automática XML ↔ diccionarios, muy cómoda cuando no se necesita una clase concreta).


In [12]:
from dataclasses import dataclass, field
from typing import List
import xml.etree.ElementTree as ET

@dataclass
class Alumno:
    id: int
    nombre: str
    nota: float

@dataclass
class ListaAlumnos:
    alumnos: List[Alumno] = field(default_factory=list)

    @classmethod
    def desde_xml(cls, ruta):
        '''XML -> objetos (equivalente a "unmarshalling")'''
        raiz = ET.parse(ruta).getroot()
        alumnos = [
            Alumno(id=int(el.get("id")),
                   nombre=el.find("nombre").text,
                   nota=float(el.find("nota").text))
            for el in raiz.findall("alumno")
        ]
        return cls(alumnos)

    def a_xml(self):
        '''Objetos -> XML (equivalente a "marshalling")'''
        raiz = ET.Element("alumnos")
        for a in self.alumnos:
            el = ET.SubElement(raiz, "alumno", id=str(a.id))
            ET.SubElement(el, "nombre").text = a.nombre
            ET.SubElement(el, "nota").text = str(a.nota)
        ET.indent(raiz, space="    ")
        return ET.tostring(raiz, encoding="unicode")

# XML -> objetos
lista = ListaAlumnos.desde_xml("alumnos.xml")
for a in lista.alumnos:
    print(f"{a.nombre} -> {a.nota}  (objeto Python: {a})")

# Objetos -> XML
print("\nObjetos serializados de nuevo a XML:\n")
print(lista.a_xml())


Ana García -> 8.5  (objeto Python: Alumno(id=1, nombre='Ana García', nota=8.5))
Luis Pérez -> 6.0  (objeto Python: Alumno(id=2, nombre='Luis Pérez', nota=6.0))
Marta Ruiz -> 9.25  (objeto Python: Alumno(id=3, nombre='Marta Ruiz', nota=9.25))

Objetos serializados de nuevo a XML:

<alumnos>
    <alumno id="1">
        <nombre>Ana García</nombre>
        <nota>8.5</nota>
    </alumno>
    <alumno id="2">
        <nombre>Luis Pérez</nombre>
        <nota>6.0</nota>
    </alumno>
    <alumno id="3">
        <nombre>Marta Ruiz</nombre>
        <nota>9.25</nota>
    </alumno>
</alumnos>


In [13]:
import xmltodict
import json

# xmltodict: forma MUY rápida de convertir XML <-> diccionario Python (y de ahí a JSON)
with open("alumnos.xml", encoding="utf-8") as f:
    datos = xmltodict.parse(f.read())

print("Diccionario Python obtenido a partir del XML:")
print(datos)

print("\nAcceso directo como si fuera un diccionario anidado:")
primer_alumno = datos["alumnos"]["alumno"][0]
print(" Primer alumno:", primer_alumno["nombre"], "->", primer_alumno["@id"])

print("\nConvertido a JSON:")
print(json.dumps(datos, indent=2, ensure_ascii=False))


Diccionario Python obtenido a partir del XML:
{'alumnos': {'alumno': [{'@id': '1', 'nombre': 'Ana García', 'nota': '8.5'}, {'@id': '2', 'nombre': 'Luis Pérez', 'nota': '6.0'}, {'@id': '3', 'nombre': 'Marta Ruiz', 'nota': '9.25'}]}}

Acceso directo como si fuera un diccionario anidado:
 Primer alumno: Ana García -> 1

Convertido a JSON:
{
  "alumnos": {
    "alumno": [
      {
        "@id": "1",
        "nombre": "Ana García",
        "nota": "8.5"
      },
      {
        "@id": "2",
        "nombre": "Luis Pérez",
        "nota": "6.0"
      },
      {
        "@id": "3",
        "nombre": "Marta Ruiz",
        "nota": "9.25"
      }
    ]
  }
}


## 7. Librerías para conversión de documentos XML a otros formatos

### 7.1. XSLT (*Extensible Stylesheet Language Transformations*)

XSLT es un lenguaje (basado a su vez en XML) que describe **transformaciones** de un documento XML a
otro formato (HTML, texto plano, otro XML...). La librería estándar de Python (`xml.etree.ElementTree`)
**no** implementa XSLT; para ello se usa la librería `lxml`, que sí lo soporta de forma completa.

Vamos a transformar nuestro `alumnos.xml` en una tabla **HTML** aplicando una hoja de estilos XSLT:


In [14]:
from pathlib import Path

xsl_alumnos = '''<?xml version="1.0" encoding="UTF-8"?>
<xsl:stylesheet version="1.0" xmlns:xsl="http://www.w3.org/1999/XSL/Transform">
    <xsl:output method="html" encoding="UTF-8"/>
    <xsl:template match="/alumnos">
        <html>
        <body>
            <h2>Listado de alumnos</h2>
            <table border="1">
                <tr><th>ID</th><th>Nombre</th><th>Nota</th></tr>
                <xsl:for-each select="alumno">
                    <tr>
                        <td><xsl:value-of select="@id"/></td>
                        <td><xsl:value-of select="nombre"/></td>
                        <td><xsl:value-of select="nota"/></td>
                    </tr>
                </xsl:for-each>
            </table>
        </body>
        </html>
    </xsl:template>
</xsl:stylesheet>
'''
Path("alumnos.xsl").write_text(xsl_alumnos, encoding="utf-8")
print("Hoja de estilos alumnos.xsl preparada.")


Hoja de estilos alumnos.xsl preparada.


In [15]:
from lxml import etree

xml_doc = etree.parse("alumnos.xml")
xsl_doc = etree.parse("alumnos.xsl")

transformador = etree.XSLT(xsl_doc)
resultado = transformador(xml_doc)

print("XML transformado a HTML mediante XSLT:\n")
print(str(resultado))


XML transformado a HTML mediante XSLT:

<html><body>
<h2>Listado de alumnos</h2>
<table border="1">
<tr>
<th>ID</th>
<th>Nombre</th>
<th>Nota</th>
</tr>
<tr>
<td>1</td>
<td>Ana García</td>
<td>8.5</td>
</tr>
<tr>
<td>2</td>
<td>Luis Pérez</td>
<td>6.0</td>
</tr>
<tr>
<td>3</td>
<td>Marta Ruiz</td>
<td>9.25</td>
</tr>
</table>
</body></html>



El mismo mecanismo (`etree.XSLT`) sirve para:
- Volcar el resultado a un fichero en lugar de imprimirlo (`resultado.write("salida.html")`).
- Transformar un árbol ya cargado y modificado en memoria (como el `Document` del apartado 6.1) de
  vuelta a un fichero XML.

### 7.2. Otras librerías habituales para convertir XML

Aunque no forman parte de la instalación estándar de Python (hay que instalarlas con `pip`), estas son
las más utilizadas en el sector para convertir entre XML y otros formatos:

| Librería | Conversión | Comentario |
|---|---|---|
| **`xmltodict`** | XML ↔ diccionario Python ↔ JSON | La más sencilla y directa; usada en el apartado 6.4 |
| **`lxml`** | XML ↔ XSLT, XPath avanzado, validación XSD | La librería XML más completa y rápida de Python (usada arriba) |
| **`dicttoxml`** | diccionario/JSON → XML | El sentido inverso a `xmltodict`, útil para generar XML desde datos ya estructurados |
| **`xmlschema`** | Validación y *binding* con esquemas XSD | Permite generar objetos Python validados contra un `.xsd`, similar a JAXB con esquema |
| **`pandas`** (`pandas.read_xml`) | XML → tabla (`DataFrame`) → CSV / Excel / JSON | Muy útil cuando el XML representa datos tabulares |

Un ejemplo típico de **XML → JSON** en un proyecto real usando `xmltodict` (ya visto arriba) sería:

```python
import xmltodict, json

with open("alumnos.xml", encoding="utf-8") as f:
    datos = xmltodict.parse(f.read())

with open("alumnos.json", "w", encoding="utf-8") as f:
    json.dump(datos, f, indent=2, ensure_ascii=False)
```


## 8. Excepciones: detección y tratamiento

Casi todas las operaciones de E/S pueden fallar (el fichero no existe, no hay permisos, el disco está
lleno, el formato es incorrecto...). Python obliga a **gestionar estos errores mediante excepciones**,
igual que Java, aunque con una diferencia importante: **en Python todas las excepciones son "no
comprobadas"** (no existe el equivalente al *checked exception* de Java): el compilador/intérprete
nunca obliga a capturar ni a declarar una excepción.

### 8.1. Jerarquía relevante para E/S

```
BaseException
 └─ Exception
     ├─ OSError                    (errores del sistema operativo — engloba la E/S)
     │   ├─ FileNotFoundError
     │   ├─ PermissionError
     │   ├─ IsADirectoryError
     │   └─ FileExistsError
     ├─ ValueError
     │   └─ (p. ej. errores de struct.unpack, int() con texto no numérico...)
     ├─ IndexError
     └─ KeyError
```

- Desde Python 3, `IOError` es un **alias** de `OSError`: son la misma excepción.
- Al no haber excepciones *checked*, la responsabilidad de decidir qué capturar recae por completo en
  el criterio del programador — con más libertad, pero también más riesgo de olvidar gestionar algún
  caso.

### 8.2. Ejemplo completo


In [16]:
from pathlib import Path

# Excepción PERSONALIZADA para una regla de negocio propia
class NotaInvalidaError(Exception):
    def __init__(self, mensaje):
        super().__init__(mensaje)

def validar_nota(nota):
    if nota < 0 or nota > 10:
        raise NotaInvalidaError(f"La nota {nota} está fuera de rango [0,10]")

# 1) FileNotFoundError: subclase de OSError
try:
    with open("no_existe.txt", "r", encoding="utf-8") as f:
        f.readline()
except FileNotFoundError as e:
    print("Capturada FileNotFoundError:", e)
except OSError as e:
    # Captura genérica: solo entraría aquí para OTROS OSError distintos de FileNotFoundError
    print("Capturada OSError genérica:", e)

# 2) with + finally: el recurso se cierra SIEMPRE (with) y además ejecutamos limpieza explícita
def procesar_fichero():
    try:
        with open("temporal.txt", "w", encoding="utf-8") as f:
            f.write("dato")
            raise OSError("fallo simulado durante el procesamiento")
    finally:
        Path("temporal.txt").unlink(missing_ok=True)   # limpieza garantizada aunque falle la escritura

try:
    procesar_fichero()
except OSError as e:
    print("Procesamiento fallido:", e)
finally:
    print("El bloque finally siempre se ejecuta (cierre de recursos, logs...)")

# 3) Excepción propia para validación de dominio
try:
    validar_nota(12.5)
except NotaInvalidaError as e:
    print("Error de validación:", e)

# 4) Excepción "de programación" (como una RuntimeException en Java): en Python tampoco es obligatorio
#    capturarla, pero es buena práctica hacerlo si el fallo es previsible
try:
    notas = [8, 6, 9]
    print(notas[5])
except IndexError as e:
    print("Capturada IndexError:", e)

# 5) Captura de VARIOS tipos de excepción a la vez (equivalente al multi-catch de Java 7+)
try:
    valor = None
    if valor is None:
        raise ValueError("Sin argumentos")
except (ValueError, TypeError) as e:
    print(f"Captura múltiple: {type(e).__name__} -> {e}")


Capturada FileNotFoundError: [Errno 2] No such file or directory: 'no_existe.txt'
Procesamiento fallido: fallo simulado durante el procesamiento
El bloque finally siempre se ejecuta (cierre de recursos, logs...)
Error de validación: La nota 12.5 está fuera de rango [0,10]
Capturada IndexError: list index out of range
Captura múltiple: ValueError -> Sin argumentos


### 8.3. Buenas prácticas al tratar excepciones de E/S

- **Capturar lo más específico primero**: si se capturan `FileNotFoundError` y `OSError` en el mismo
  bloque, `FileNotFoundError` debe ir en el primer `except` (Python evalúa los `except` en orden y
  usa el primero que coincida, igual que Java).
- **No "tragarse" excepciones** con un `except: pass`: como mínimo, registrar el error (`logging`) para
  poder depurar el problema.
- **Usar siempre `with`** al trabajar con ficheros o cualquier recurso que deba cerrarse, en lugar de
  llamar a `close()` manualmente en un `finally`.
- **Crear excepciones propias** (como `NotaInvalidaError`) heredando de `Exception`, cuando se
  detecten errores de **reglas de negocio**, para diferenciarlos claramente de los errores técnicos de
  E/S (`OSError`).
- **Usar `raise ... from e`** al relanzar una excepción distinta dentro de un `except`, para conservar
  la traza original y facilitar la depuración (encadenamiento de excepciones).
- **Evitar el patrón *LBYL* (*Look Before You Leap*)** en exceso (comprobar `if fichero.exists()` antes
  de abrir) a favor de *EAFP* (*Easier to Ask Forgiveness than Permission*: abrir directamente y
  capturar la excepción si falla), que es el estilo idiomático en Python y evita condiciones de
  carrera entre la comprobación y el uso real del fichero.


## Resumen y ejercicios propuestos

### Resumen de módulos/funciones vistas

| Tema | Python | Equivalente en Java |
|---|---|---|
| Gestión de ficheros/directorios | `pathlib.Path`, `os`, `os.path`, `shutil` | `java.io.File`, `java.nio.file.Path`/`Files` |
| Acceso aleatorio | `f.seek()` / `f.tell()` sobre cualquier fichero abierto | `java.io.RandomAccessFile` |
| Flujos de bytes | `open(ruta, "rb"/"wb")` | `FileInputStream` / `FileOutputStream` |
| Flujos de caracteres | `open(ruta, "r"/"w", encoding=...)` | `FileReader` / `FileWriter` |
| Datos binarios con formato | `struct.pack()` / `struct.unpack()` | `DataOutputStream` / `DataInputStream` |
| Serialización de objetos | `pickle.dump()` / `pickle.load()` | `ObjectOutputStream` / `ObjectInputStream` |
| XML — parsers | `xml.dom.minidom` (DOM), `xml.sax` (SAX), `xml.etree.ElementTree` (StAX-like) | `DocumentBuilder`, `SAXParser`, `XMLStreamReader` |
| XML — binding | `dataclasses` + `ElementTree`, `xmltodict` | `JAXBContext`, `Marshaller`, `Unmarshaller` |
| XML — conversión | `lxml.etree.XSLT`, `xmltodict` + `json` | `javax.xml.transform.Transformer`, Jackson |
| Excepciones | `OSError`, `FileNotFoundError`, excepciones propias (`class X(Exception)`) | `IOException`, `FileNotFoundException`, excepciones propias |

### Ejercicios propuestos

1. Modifica el ejemplo de `pathlib` para que, antes de borrar el directorio `datos_demo`, compruebe
   con un `if` si está vacío y, si no lo está, borre primero todos los ficheros que contiene (o usa
   directamente `shutil.rmtree`, y explica en un comentario la diferencia entre ambos enfoques).
2. Amplía el ejemplo de acceso aleatorio añadiendo un cuarto registro y una función
   `buscar_por_nombre` que recorra todos los registros hasta encontrar uno con un nombre dado (esto
   muestra que, aunque el acceso sea "aleatorio", **buscar por un campo que no sea la posición sigue
   requiriendo recorrer los datos**, salvo que se mantenga un índice aparte).
3. Escribe un programa que lea un fichero de texto y cuente cuántas líneas, palabras y caracteres
   tiene (como el comando `wc` de Linux), usando `for linea in f:`.
4. Añade al XML `alumnos.xml` un cuarto alumno y adapta el ejemplo de SAX para que, además de la
   media, calcule la nota **máxima** y **mínima**.
5. Modifica la hoja de estilos `alumnos.xsl` para que, en lugar de una tabla HTML, genere un documento
   de texto plano con el formato `Nombre: X — Nota: Y`, uno por línea (pista: cambia
   `<xsl:output method="html"/>` por `method="text"` y ajusta la plantilla).
6. Crea una excepción propia `FicheroDemasiadoGrandeError` y lánzala desde una función que compruebe,
   antes de leer un fichero, si su tamaño (`Path.stat().st_size`) supera un límite dado.
